<a href="https://colab.research.google.com/github/0MSA0/supreme-robot/blob/main/data_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Data-Loader

DataSet Class:

In [1]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

class SupremeDataset(Dataset):
  def __init__(self, input_text: str, tokenizer, max_length, stride) -> None:
    self.input_ids = []
    self.target_ids = []

    encoded_words = tokenizer.encode(input_text)

    # Use a sliding window to chunk the input text
    # into overlapping sequences of max_length
    for i in range(0, len(encoded_words) - max_length, stride):
      input_chunk = encoded_words[i:i + max_length]
      target_chunk = encoded_words[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]


DataLoader:

In [2]:
def create_supreme_dataloader(txt, tokenizer, batch_size=4, max_length=256,
                              stride=128, shuffle=True, drop_last=True,
                              num_workers=0):

    # Create dataset
    dataset = SupremeDataset(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

Function for creating the training data based on a text:

In [6]:
def create_training_data(raw_text, tokenizer, batch_size=8,
                        max_length=4, stride=4):
  dataloader = create_supreme_dataloader(
      raw_text,
      tokenizer,
      batch_size=batch_size,
      max_length=max_length,
      stride=stride
  )

  # Input tokens
  input_token_ids_list = []
  # Expected target tokens
  target_token_ids_list = []

  for batch in dataloader:
      input_token_ids, target_token_ids = batch
      input_token_ids_list.append(input_token_ids)
      target_token_ids_list.append(target_token_ids)

  return input_token_ids_list, target_token_ids_list

Open sample text:

In [7]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

Create training-data based on the for sample text:

In [8]:
tokenizer = tiktoken.get_encoding("o200k_base")
input_token_ids_list, target_token_ids_list = create_training_data(raw_text,
                                                                  tokenizer)
print(input_token_ids_list[0])
print(target_token_ids_list[0])

tensor([[  6787,   5868,    267,  10701],
        [   375, 160689,  12183,     11],
        [   480,  16842,    357,    673],
        [  2395,     11,   7892,    357],
        [ 11723,  53748,   5861,     13],
        [ 44599,    484,  90089,    290],
        [  1261,     11,    402,    261],
        [   316,    668,    316,   6213]])
tensor([[  5868,    267,  10701,    540],
        [160689,  12183,     11,  13434],
        [ 16842,    357,    673,   1078],
        [    11,   7892,    357,   6375],
        [ 53748,   5861,     13,    392],
        [   484,  90089,    290,  18451],
        [    11,    402,    261,   4849],
        [   668,    316,   6213,   4436]])
